# 05 — Validation and parallelism

<sup>(C) Copyright 2026- ECMWF and individual contributors. Licensed under the Apache Licence, Version 2.0.</sup>

**What you will learn**

- Run the four levels of `tensogram.validate()` — structure, metadata, integrity, fidelity — and see the issue codes they emit.
- Inject two kinds of corruption (truncation and xxh3 mismatch) and watch validation detect them.
- Scale encoding and decoding across multiple threads with the `threads=N` option (new in v0.13).
- Understand Tensogram's determinism contract: transparent codecs produce byte-identical output across thread counts; opaque codecs (blosc2, zstd with workers) round-trip losslessly but may reorder internal blocks.

**Prerequisites**: see [`README.md`](README.md). No OS-level C libraries needed for this notebook.

In [ ]:
import matplotlib

matplotlib.use("Agg")

import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np

import tensogram

print(f"cpu count: {os.cpu_count()}")

Build a synthetic "weather field" big enough that parallelism has something to chew on (~16 MiB of float64).

In [ ]:
NPOINTS = 2_000_000  # ~16 MiB of float64
rng = np.random.default_rng(seed=42)
# Smooth low-entropy field that compresses well.
x = np.linspace(0, 4 * np.pi, NPOINTS)
raw = (273.15 + 15 * np.sin(x) + 0.3 * rng.standard_normal(NPOINTS)).astype(np.float64)
print(f"array size: {raw.nbytes / 1024 / 1024:.2f} MiB")


def make_message(values, *, threads=0):
    """Encode with simple_packing + zstd (transparent codecs → fully deterministic)."""
    params = tensogram.compute_packing_params(
        values.astype(np.float64).ravel(),
        bits_per_value=16,
        decimal_scale_factor=0,
    )
    desc = {
        "type": "ntensor", "shape": list(values.shape), "dtype": "float64",
        "encoding": "simple_packing", "filter": "none",
        "compression": "zstd", "zstd_level": 3,
        **params,
    }
    return bytes(tensogram.encode({"version": 2}, [(desc, values)],
                                   threads=threads))


message = make_message(raw)
print(f"encoded message: {len(message):,} bytes "
      f"({len(message) / raw.nbytes * 100:.2f}% of raw)")

## 1. `tensogram.validate()` — the four levels

The validator runs in levels:

| Level | Name | Cost | Catches |
|---|---|---|---|
| 1 | **Structure** | cheap | preamble magic, frame headers, total_length, trailer marker, frame order, overflow safety |
| 2 | **Metadata** | cheap | required CBOR keys, dtype/encoding/compression names, shape/strides consistency, index+hash frame consistency |
| 3 | **Integrity** | medium | xxh3 hash verification, decode round-trip for compressed objects |
| 4 | **Fidelity** | expensive | full decode + decoded-size check + NaN/Inf scan |

Levels 1–3 are the default. Level 4 (`level="full"`) adds the fidelity scan. There's also `level="quick"` (level 1 only) and `level="checksum"` (levels 1+2+checksum).

In [ ]:
def is_valid(report):
    """No `error`-severity issues means the message is valid."""
    return not any(i["severity"] == "error" for i in report["issues"])


for level in ("quick", "checksum", "default", "full"):
    report = tensogram.validate(message, level=level)
    print(f"level={level:<10} valid={is_valid(report)}  "
          f"issues={len(report['issues'])}  "
          f"hash_verified={report.get('hash_verified', None)}")

## 2. Injecting corruption

Let's see what the validator says when things go wrong.

### 2a. Truncation — cut the message in half

A truncated message has a preamble but no footer. Level 1 (structure) should catch it immediately.

In [ ]:
truncated = message[: len(message) // 2]
report = tensogram.validate(truncated, level="quick")
print(f"valid: {is_valid(report)}")
for issue in report["issues"][:5]:
    print(f"  [{issue['severity']:<7}] {issue['code']:<40}  {issue['description']}")

### 2b. Payload bit-flip — xxh3 mismatch

Flip a byte in the middle of the payload. Structure and metadata are still valid; Level 3 (integrity) catches the corruption via the stored hash.

In [ ]:
corrupted = bytearray(message)
# Flip a byte roughly in the middle of the payload (past the header frames).
corrupted[len(corrupted) // 2] ^= 0x55
corrupted = bytes(corrupted)

# Level 1 (structure) passes because frame headers are untouched:
r1 = tensogram.validate(corrupted, level="quick")
print(f"level=quick    valid={is_valid(r1)}  issues={len(r1['issues'])}")

# Level 3 (integrity) catches the hash mismatch:
r3 = tensogram.validate(corrupted, level="default")
print(f"level=default  valid={is_valid(r3)}  "
      f"issues={len(r3['issues'])}  "
      f"hash_verified={r3.get('hash_verified', None)}")
for issue in r3["issues"][:3]:
    print(f"  [{issue['severity']:<7}] {issue['code']:<40}  {issue['description']}")

## 3. Multi-threaded encoding and decoding

`tensogram.encode(..., threads=N)` and `tensogram.decode(..., threads=N)` spread the encoding pipeline across `N` rayon workers. `threads=0` is sequential (the default). The `TENSOGRAM_THREADS` environment variable is used when `threads=0`.

For transparent codecs (none, simple_packing, shuffle, lz4, szip, zfp, sz3) the encoded bytes are **byte-identical** across thread counts. For opaque codecs (blosc2, zstd-with-workers) the bytes may reorder but the decoded values are identical.

In [ ]:
# Sweep threads = 0, 1, 2, 4 on encode.
thread_counts = [0, 1, 2, 4]
encode_ms, decode_ms = [], []
sizes = []

WARMUP = 1
REPEATS = 3

for n in thread_counts:
    for _ in range(WARMUP):
        _ = make_message(raw, threads=n)

    t_enc = []
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        msg = make_message(raw, threads=n)
        t_enc.append((time.perf_counter() - t0) * 1000)

    t_dec = []
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        _ = tensogram.decode(msg, threads=n)
        t_dec.append((time.perf_counter() - t0) * 1000)

    encode_ms.append(min(t_enc))
    decode_ms.append(min(t_dec))
    sizes.append(len(msg))
    print(f"threads={n}  enc={min(t_enc):6.1f}ms  dec={min(t_dec):6.1f}ms  size={len(msg):,} B")

In [ ]:
# Plot speedup vs threads.
fig, ax = plt.subplots(figsize=(7, 4))
enc_speedup = [encode_ms[0] / m for m in encode_ms]
dec_speedup = [decode_ms[0] / m for m in decode_ms]

x_labels = [f"threads={n}" for n in thread_counts]
ax.plot(x_labels, enc_speedup, "o-", label="encode", linewidth=2)
ax.plot(x_labels, dec_speedup, "s--", label="decode", linewidth=2)
ax.axhline(1.0, color="gray", linestyle=":", alpha=0.5)
ax.set_ylabel("speedup vs threads=0")
ax.set_title(f"Parallel scaling (simple_packing 16-bit + zstd, {raw.nbytes // (1024*1024)} MiB field)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Determinism across thread counts

For the **transparent** codecs we used above (`simple_packing` is transparent; `zstd` becomes opaque only when `NbWorkers > 0`), the encoded bytes must be byte-identical regardless of thread count. Let's verify.

In [ ]:
bytes_by_threads = {}
for n in [0, 1, 2, 4]:
    bytes_by_threads[n] = make_message(raw, threads=n)

# The `_reserved_.time` + `_reserved_.uuid` fields will make the full bytes
# vary per call. Decoded payloads should be identical though — that's
# the real determinism contract.
ref = bytes_by_threads[0]
_, ref_objs = tensogram.decode(ref)
ref_values = ref_objs[0][1]

for n, msg in bytes_by_threads.items():
    _, objs = tensogram.decode(msg)
    np.testing.assert_array_equal(ref_values, objs[0][1])
    print(f"threads={n} decoded payload matches threads=0 reference "
          f"(byte-level diff: {sum(1 for a, b in zip(ref, msg) if a != b):5d} bytes)")

print("\ndetermined: decoded payloads are byte-identical across thread counts")

## 5. Validation + parallelism in CI

Putting it all together for an end-to-end pipeline:

```python
# Encode with as much parallelism as the host can spare.
import os
messages = tensogram.convert_netcdf(
    "huge.nc",
    compression="zstd",
    threads=os.cpu_count() or 4,
)

# Validate *before* serving or storing:
for msg in messages:
    report = tensogram.validate(msg, level="default")
    if not report["valid"]:
        raise ValueError(f"invalid tensogram: {report['issues']}")
```

## Where to go next

- [`../python/16_multi_threaded_pipeline.py`](../python/16_multi_threaded_pipeline.py) — the standalone Python example with more benchmarking knobs.
- [`../../docs/src/guide/multi-threaded-pipeline.md`](../../docs/src/guide/multi-threaded-pipeline.md) — reference for the full `threads` API and determinism contract.
- [`../../docs/src/cli/validate.md`](../../docs/src/cli/validate.md) — the `tensogram validate` CLI (same 4 levels, exposed via the command line and `--json` output).